# 02 — SFT warmup

**Purpose.** Fine-tune `Qwen/Qwen2.5-Coder-3B-Instruct` (or any compatible base) on the trajectories produced by `01_data_mining.ipynb` so the model learns the Graft tool-calling protocol before RL.

**Inputs.** `data/trajectories.jsonl` (one JSON object per line; see notebook 01).

**Outputs.** A LoRA adapter at `training/checkpoints/sft/` and a small held-out evaluation report.

**Knobs.** `BASE_MODEL`, `EPOCHS`, `BATCH_SIZE`, `LR`, `LORA_R`. Defaults match the build spec.

In [ ]:
!pip install trl
# ---------- setup: ensure torchao is compatible ----------
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-U",
    "torchao>=0.16.0",
])

In [ ]:
# ---------- CONFIG ----------
import os
from getpass import getpass
from pathlib import Path

BASE_MODEL          = os.environ.get("TRAINING_BASE_MODEL", "Qwen/Qwen2.5-Coder-3B-Instruct")
DATASET_NAME        = "devaanshpa/graft-small-dataset"
TRAJECTORIES_FILE   = "trajectories.jsonl"
OUTPUT_DIR          = Path("./checkpoints/sft")
EPOCHS              = 3
BATCH_SIZE          = 2           # T4 = 16 GB; bs=2 + seq=2048 is safe with gradient checkpointing
GRAD_ACCUM          = 16          # effective batch size = 32
LR                  = 2e-5
MAX_SEQ_LEN         = 2048        # 4096 OOMs on T4 at bs=2; 2048 is safe
WARMUP_RATIO        = 0.05
LOGGING_STEPS       = 1           # print every step
EVAL_STEPS          = 10
LORA_R              = 16
LORA_ALPHA          = 32
VAL_FRACTION        = 0.1
SEED                = 42

HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
if not HF_TOKEN:
    HF_TOKEN = getpass("HF token (leave blank to skip uploads): ").strip()
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

In [ ]:
# ---------- imports ----------
import json
import random
import matplotlib.pyplot as plt
import torch
from datasets import Dataset, load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback
from trl import SFTConfig, SFTTrainer

random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# ---------- load trajectories ----------
ds = load_dataset(DATASET_NAME, data_files=TRAJECTORIES_FILE, split="train")
records = [r for r in ds]
print(f"Loaded {len(records)} trajectories from {DATASET_NAME}/{TRAJECTORIES_FILE}")
random.shuffle(records)
val_n = max(1, int(len(records) * VAL_FRACTION))
train_records = records[val_n:]
val_records = records[:val_n]
print(f"Train: {len(train_records)} | Val: {len(val_records)}")

In [ ]:
# ---------- tokenize via the model's chat template ----------
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def render(record):
    msgs = []
    for m in record["messages"]:
        role = m["role"]
        if role == "tool":
            msgs.append({
                "role": "tool",
                "content": m.get("content", ""),
            })
        elif role == "assistant" and m.get("tool_calls"):
            content = ""
            for tc in m["tool_calls"]:
                content += json.dumps({"tool": tc["name"], "args": tc.get("args", {})}) + "\n"
            msgs.append({"role": "assistant", "content": content.strip()})
        else:
            msgs.append({"role": role, "content": m.get("content", "") or ""})
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    return {"text": text}

train_ds = Dataset.from_list([render(r) for r in train_records])
val_ds = Dataset.from_list([render(r) for r in val_records])
print("Sample (truncated):\n", train_ds[0]["text"][:600])

In [ ]:
# ---------- model + LoRA + trainer ----------
import math

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16
dtype = torch.bfloat16 if use_bf16 else (torch.float16 if use_fp16 else torch.float32)
print(f"Precision: {'bfloat16' if use_bf16 else 'float16' if use_fp16 else 'float32'}", flush=True)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)
model.config.use_cache = False

lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

steps_per_epoch = math.ceil(len(train_ds) / max(1, BATCH_SIZE * GRAD_ACCUM))
total_steps = max(1, steps_per_epoch * EPOCHS)
warmup_steps = max(1, int(total_steps * WARMUP_RATIO))
print(f"Steps/epoch={steps_per_epoch} | total_steps={total_steps} | warmup={warmup_steps}", flush=True)

sft_cfg = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    logging_steps=1,
    logging_first_step=True,
    logging_strategy="steps",
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=EVAL_STEPS,
    save_total_limit=2,
    bf16=use_bf16,
    fp16=use_fp16,
    gradient_checkpointing=True,
    max_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    seed=SEED,
    report_to=[],
    disable_tqdm=True,
)


class StepLogger(TrainerCallback):
    # on_log fires AFTER the Trainer computes metrics and receives them as `logs` directly.
    # on_step_end fires BEFORE on_log so log_history is still stale — don't use it.
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        parts = [f"step={state.global_step}/{state.max_steps}"]
        for k, v in sorted(logs.items()):
            parts.append(f"{k}={v:.4f}" if isinstance(v, float) else f"{k}={v}")
        print(" | ".join(parts), flush=True)


class LossCollector(TrainerCallback):
    def __init__(self):
        self.steps, self.losses, self.eval_losses = [], [], []
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            self.steps.append(state.global_step)
            self.losses.append(logs["loss"])
        if logs and "eval_loss" in logs:
            self.eval_losses.append((state.global_step, logs["eval_loss"]))


loss_cb = LossCollector()

trainer = SFTTrainer(
    model=model,
    args=sft_cfg,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=lora_cfg,
    processing_class=tokenizer,
    callbacks=[loss_cb, StepLogger()],
)
trainer.train()

In [ ]:
# ---------- loss curve ----------
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(loss_cb.steps, loss_cb.losses, label="train loss", color="#2563eb")
if loss_cb.eval_losses:
    es, ev = zip(*loss_cb.eval_losses)
    ax.plot(es, ev, marker="o", color="#f59e0b", label="eval loss")
ax.set_xlabel("step"); ax.set_ylabel("loss"); ax.legend(); ax.set_title("SFT loss")
plt.tight_layout(); plt.show()

In [ ]:
# ---------- save adapter + tokenizer ----------
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"SFT adapter saved to {OUTPUT_DIR}")
print("Files:", sorted(p.name for p in OUTPUT_DIR.iterdir()))

In [ ]:
# ---------- upload SFT adapter to Hugging Face model repo ----------
from pathlib import Path
from huggingface_hub import login, upload_folder

repo_id = "devaanshpa/Qwen2.5-Coder-3B-Instruct-Graft"
upload_dir = OUTPUT_DIR if "OUTPUT_DIR" in globals() else Path("./checkpoints/sft")
if not upload_dir.exists():
    raise FileNotFoundError(f"Upload directory not found: {upload_dir}")

if not HF_TOKEN:
    print("No HF token — skipping model repo upload.", flush=True)
else:
    login(token=HF_TOKEN)
    upload_folder(folder_path=str(upload_dir), repo_id=repo_id, repo_type="model")

In [ ]:
# ---------- quick eval: 10 held-out trajectories ----------
import re

def _expected_tools(record) -> list[str]:
    tools = []
    for m in record["messages"]:
        if m["role"] == "assistant" and m.get("tool_calls"):
            tools.extend(tc["name"] for tc in m["tool_calls"])
    return tools

def _generate(prompt_msgs):
    text = tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(trainer.model.device)
    with torch.no_grad():
        out = trainer.model.generate(**inputs, max_new_tokens=512, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

heldout = val_records[:10]
hits = 0
for rec in heldout:
    expected = _expected_tools(rec)
    if not expected:
        continue
    prompt = [m for m in rec["messages"] if m["role"] in ("system", "user")][:2]
    text = _generate(prompt)
    first_tool = None
    m = re.search(r'"tool"\s*:\s*"([a-z_]+)"', text)
    if m:
        first_tool = m.group(1)
    if first_tool == expected[0]:
        hits += 1
print(f"First-tool match on heldout: {hits}/{len(heldout)}")

In [ ]:
# ---------- sync /kaggle/working to HF Bucket ----------
# hf_xet is a PyO3 native extension — once it errors in this process it can't be
# re-imported. Running sync in a fresh subprocess avoids that completely.
import os, subprocess, sys

BUCKET = "devaanshpa/Graft-SFT-Output"

if not HF_TOKEN:
    print("No HF token — skipping bucket sync.", flush=True)
else:
    print(f"Syncing /kaggle/working → hf://buckets/{BUCKET} ...", flush=True)
    _script = (
        "import os; from huggingface_hub import sync_bucket, login; "
        "login(token=os.environ['HF_TOKEN']); "
        f"sync_bucket('/kaggle/working', 'hf://buckets/{BUCKET}', "
        "exclude=['**/__pycache__/**', '**/*.pyc', '**/.git/**']); "
        "print('Bucket sync complete.', flush=True)"
    )
    subprocess.check_call(
        [sys.executable, "-c", _script],
        env={**os.environ, "HF_TOKEN": HF_TOKEN},
    )